In [21]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# Paths
INPUT_DIR = "Data\For Enhancement and Filtering"
ENH_OUTPUT_DIR = "output/enhancement/"
os.makedirs(ENH_OUTPUT_DIR, exist_ok=True)

# List all image files (modify extensions as needed)
image_files = [f for f in os.listdir(INPUT_DIR) 
               if f.endswith(('.jpg','.png','.jpeg')) and not os.path.isdir(os.path.join(INPUT_DIR,f))]

def histogram_equalization(img_gray):
    """Apply global histogram equalization."""
    return cv2.equalizeHist(img_gray)

def gamma_correction(img_gray, gamma=1.5):
    """Adjust gamma (gamma >1 brightens midtones)."""
    inv_gamma = 1.0 / gamma
    table = np.array([(i / 255.0) ** inv_gamma * 255 for i in range(256)]).astype(np.uint8)
    return cv2.LUT(img_gray, table)

# Process first 4 images for report examples
example_images = image_files[:4]

for idx, img_name in enumerate(example_images):
    img_path = os.path.join(INPUT_DIR, img_name)
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)  # read as grayscale
    
    # Apply both techniques
    img_histeq = histogram_equalization(img)
    img_gamma = gamma_correction(img, gamma=1.8)
    
    # Save enhanced images
    base_name = os.path.splitext(img_name)[0]
    cv2.imwrite(os.path.join(ENH_OUTPUT_DIR, f"{base_name}_histeq.jpg"), img_histeq)
    cv2.imwrite(os.path.join(ENH_OUTPUT_DIR, f"{base_name}_gamma.jpg"), img_gamma)
    
    # Plot and save histograms for histeq (you can also do for gamma)
    plt.figure(figsize=(12,4))
    plt.subplot(1,3,1); plt.imshow(img, cmap='gray'); plt.title('Original'); plt.axis('off')
    plt.subplot(1,3,2); plt.imshow(img_histeq, cmap='gray'); plt.title('Histogram Equalized'); plt.axis('off')
    plt.subplot(1,3,3); plt.hist(img.ravel(),256,[0,256], alpha=0.6, label='Original')
    plt.hist(img_histeq.ravel(),256,[0,256], alpha=0.6, label='Equalized')
    plt.legend(); plt.title('Histogram Comparison')
    plt.tight_layout()
    plt.savefig(os.path.join(ENH_OUTPUT_DIR, f"{base_name}_hist_comparison.png"))
    plt.close()

C:\Users\muham\AppData\Local\Temp\ipykernel_14596\1695877440.py:45: MatplotlibDeprecationWarning: Passing the range parameter of hist() positionally is deprecated since Matplotlib 3.9; the parameter will become keyword-only in 3.11.
  plt.subplot(1,3,3); plt.hist(img.ravel(),256,[0,256], alpha=0.6, label='Original')
C:\Users\muham\AppData\Local\Temp\ipykernel_14596\1695877440.py:46: MatplotlibDeprecationWarning: Passing the range parameter of hist() positionally is deprecated since Matplotlib 3.9; the parameter will become keyword-only in 3.11.
  plt.hist(img_histeq.ravel(),256,[0,256], alpha=0.6, label='Equalized')


In [22]:
FILTER_OUTPUT_DIR = "output/filtering/"
os.makedirs(FILTER_OUTPUT_DIR, exist_ok=True)

for idx, img_name in enumerate(example_images):
    base_name = os.path.splitext(img_name)[0]
    # Load the enhanced image (we use histogram equalized from Stage 1)
    enhanced_path = os.path.join(ENH_OUTPUT_DIR, f"{base_name}_histeq.jpg")
    if not os.path.exists(enhanced_path):
        continue
    img_enh = cv2.imread(enhanced_path, cv2.IMREAD_GRAYSCALE)
    
    # Median filter (kernel size 5)
    median_filtered = cv2.medianBlur(img_enh, 5)
    
    # Bilateral filter (diameter=9, sigma_color=75, sigma_space=75)
    bilateral_filtered = cv2.bilateralFilter(img_enh, 9, 75, 75)
    
    # Save results
    cv2.imwrite(os.path.join(FILTER_OUTPUT_DIR, f"{base_name}_median.jpg"), median_filtered)
    cv2.imwrite(os.path.join(FILTER_OUTPUT_DIR, f"{base_name}_bilateral.jpg"), bilateral_filtered)
    
    # Create side‑by‑side comparison figure
    plt.figure(figsize=(12,4))
    plt.subplot(1,3,1); plt.imshow(img_enh, cmap='gray'); plt.title('Enhanced (histeq)'); plt.axis('off')
    plt.subplot(1,3,2); plt.imshow(median_filtered, cmap='gray'); plt.title('Median Filter (k=5)'); plt.axis('off')
    plt.subplot(1,3,3); plt.imshow(bilateral_filtered, cmap='gray'); plt.title('Bilateral Filter'); plt.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(FILTER_OUTPUT_DIR, f"{base_name}_filter_comparison.png"))
    plt.close()

In [20]:
EDGE_INPUT_DIR = os.path.join(INPUT_DIR, "For Edge Detection")   # adjust if folder name differs
EDGE_OUTPUT_DIR = "output/edge_detection/"
os.makedirs(EDGE_OUTPUT_DIR, exist_ok=True)

# List images in edge detection folder
if os.path.exists(EDGE_INPUT_DIR):
    edge_images = [f for f in os.listdir(EDGE_INPUT_DIR) if f.endswith(('.jpg','.png','.jpeg'))]
else:
    print("Edge detection folder not found. Adjust the path.")
    edge_images = []

# Take first 4 edge images
edge_examples = edge_images[:4]

for img_name in edge_examples:
    img_path = os.path.join(EDGE_INPUT_DIR, img_name)
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    
    # Canny with two thresholds (you may tune)
    edges_canny = cv2.Canny(img, 50, 150)
    
    # Optional: Sobel (gradient magnitude)
    sobelx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = np.sqrt(sobelx**2 + sobely**2)
    sobel_mag = np.uint8(sobel_mag / np.max(sobel_mag) * 255)
    
    base = os.path.splitext(img_name)[0]
    cv2.imwrite(os.path.join(EDGE_OUTPUT_DIR, f"{base}_canny.jpg"), edges_canny)
    cv2.imwrite(os.path.join(EDGE_OUTPUT_DIR, f"{base}_sobel.jpg"), sobel_mag)
    
    # Display original and edges
    plt.figure(figsize=(12,4))
    plt.subplot(1,3,1); plt.imshow(img, cmap='gray'); plt.title('Original'); plt.axis('off')
    plt.subplot(1,3,2); plt.imshow(edges_canny, cmap='gray'); plt.title('Canny Edges'); plt.axis('off')
    plt.subplot(1,3,3); plt.imshow(sobel_mag, cmap='gray'); plt.title('Sobel Magnitude'); plt.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(EDGE_OUTPUT_DIR, f"{base}_edge_comparison.png"))
    plt.close()